In [5]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
import statsmodels.api as sm
# From FPP3 book (The Pythonic Way)
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*FigureCanvasAgg is non-interactive.*"
)
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)
import random
random.seed(1)
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)
from utilsforecast.plotting import plot_series as plot_series_utils
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})
import matplotlib as mpl
from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])
from fpppy.utils import plot_series

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#000000"], name="black"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#569CC6", "#D55F03"],
        name='black_and_2color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55F03", "#569CC6", "#13A076"],
        name='black_and_3color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55F03", "#569CC6", "#13A076", "#CC79A7"],
        name='black_and_4color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#D55F03", "#569CC6", "#13A076", "#CC79A7"],
        name='r_colors',
    ),
    force=True
)

C:\Users\tanwa\AppData\Local\Temp\ipykernel_18308\1293146190.py:45: UserWarning: Overwriting the cmap 'black_and_blue' that was already in the registry.
  mpl.colormaps.register(
C:\Users\tanwa\AppData\Local\Temp\ipykernel_18308\1293146190.py:50: UserWarning: Overwriting the cmap 'black_and_orange' that was already in the registry.
  mpl.colormaps.register(
C:\Users\tanwa\AppData\Local\Temp\ipykernel_18308\1293146190.py:55: UserWarning: Overwriting the cmap 'black' that was already in the registry.
  mpl.colormaps.register(
C:\Users\tanwa\AppData\Local\Temp\ipykernel_18308\1293146190.py:60: UserWarning: Overwriting the cmap 'black_and_2color' that was already in the registry.
  mpl.colormaps.register(
C:\Users\tanwa\AppData\Local\Temp\ipykernel_18308\1293146190.py:67: UserWarning: Overwriting the cmap 'black_and_3color' that was already in the registry.
  mpl.colormaps.register(
C:\Users\tanwa\AppData\Local\Temp\ipykernel_18308\1293146190.py:74: UserWarning: Overwriting the cmap 'black

In [6]:
import matplotlib.dates as matplotdates
import matplotlib.ticker as ticker
from functools import partial, reduce

import statsmodels.api as sm
from matplotlib.ticker import MaxNLocator
from prophet import Prophet
from statsforecast import StatsForecast
from statsforecast.adapters.prophet import AutoARIMAProphet
from statsforecast.models import MSTL, AutoETS, AutoARIMA, ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.api import VAR
from utilsforecast.evaluation import evaluate
from utilsforecast.feature_engineering import trend, fourier, pipeline
from utilsforecast.losses import rmse, mae, mape, mase, smape
from utilsforecast.preprocessing import fill_gaps
import time
import holidays

## Loading the Dataset

In [27]:
talinn_data = pd.read_csv("./TallinnData_Cleaned_Weather.csv")

In [28]:
talinn_data.head(3)

,id,andurid_id,device,utc timestamp,count,in,out,date,day_of_week,month,...,humidity,precipitation,rain,snowfall,snow_depth,wind_speed,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high
0,177571,-1,Suur-Karja 18,2021-12-03 10:39:34+00:00,1.0,0.0,1.0,2021-12-03,4,12,...,81,0.3,0.0,0.21,0.04,21.6,100,98,100,88
1,76338,-1,Suur-Karja 18,2021-12-03 10:59:43+00:00,10.0,7.0,3.0,2021-12-03,4,12,...,81,0.3,0.0,0.21,0.04,21.6,100,98,100,88
2,755740,-2,Väike-Karja 12,2021-12-05 10:24:00+00:00,107.0,0.0,0.0,2021-12-05,6,12,...,83,0.1,0.0,0.07,0.05,6.8,100,100,76,0


In [ ]:
# ptc_weather = pd.read_csv("./data/ptc_weather.csv")
# ptc_weather["ds"] = pd.to_datetime(ptc_weather["ds"])

In [9]:
ptc_weather = pd.read_csv("./kafka_coding_2/data/ptc_weather.csv")
ptc_weather["ds"] = pd.to_datetime(ptc_weather["ds"])


## Loading the Estonian Holidays

In [13]:
# Adding in the holidays in Estonia as marked by the Holidays library
ee_holidays = holidays.Estonia(years=[2022,2023])
ee_holidays

{datetime.date(2022, 1, 1): 'uusaasta', datetime.date(2022, 2, 24): 'iseseisvuspäev', datetime.date(2022, 4, 15): 'suur reede', datetime.date(2022, 4, 17): 'ülestõusmispühade 1. püha', datetime.date(2022, 6, 5): 'nelipühade 1. püha', datetime.date(2022, 5, 1): 'kevadpüha', datetime.date(2022, 6, 23): 'võidupüha', datetime.date(2022, 6, 24): 'jaanipäev', datetime.date(2022, 8, 20): 'taasiseseisvumispäev', datetime.date(2022, 12, 24): 'jõululaupäev', datetime.date(2022, 12, 25): 'esimene jõulupüha', datetime.date(2022, 12, 26): 'teine jõulupüha', datetime.date(2023, 1, 1): 'uusaasta', datetime.date(2023, 2, 24): 'iseseisvuspäev', datetime.date(2023, 4, 7): 'suur reede', datetime.date(2023, 4, 9): 'ülestõusmispühade 1. püha', datetime.date(2023, 5, 28): 'nelipühade 1. püha', datetime.date(2023, 5, 1): 'kevadpüha', datetime.date(2023, 6, 23): 'võidupüha', datetime.date(2023, 6, 24): 'jaanipäev', datetime.date(2023, 8, 20): 'taasiseseisvumispäev', datetime.date(2023, 12, 24): 'jõululaupäev'

## Creating the datasets I'm using for this dirty version

In [14]:
# STEP 1: Set the true data that we're going to pass every hour to re-predict! Or maybe every 8 hours, say...
ptc_weather_2022 = ptc_weather[ptc_weather["year"] == 2022].copy()
# Here we set the holidays
ptc_weather_2022["is_holiday"] = ptc_weather_2022["ds"].dt.date.map(
    lambda x : 1 if x in ee_holidays else 0
)
# Okay, now let's drop whatever we don't need
ptc_weather_2022 = ptc_weather_2022.drop(
    columns=[
        'year', 'month', 'day', 'hour', 'in_sum', 'out_sum', #'day_of_week',
    ]
)
ptc_weather_2022.head(3)

,unique_id,ds,y,day_of_week,temp_mean,humidity_mean,precipitation_mean,rain_mean,snowfall_mean,snow_depth_mean,wind_speed_mean,cloud_cover_mean,cloud_cover_low_mean,cloud_cover_mid_mean,cloud_cover_high_mean,is_holiday
0,all_sensors,2022-01-01 00:00:00,4852.0,5,-0.6,97.0,0.2,0.1,0.07,0.09,9.4,100.0,100.0,100.0,0.0,1
1,all_sensors,2022-01-01 01:00:00,4175.0,5,-0.1,98.0,0.3,0.0,0.21,0.09,4.7,100.0,100.0,100.0,83.0,1
2,all_sensors,2022-01-01 02:00:00,4016.0,5,-0.3,99.0,0.0,0.0,0.00,0.09,4.3,100.0,100.0,100.0,21.0,1


In [15]:
# This "dev" dataset we can use to progressively feed the data with new entries. It would be better to use ptc_weather, now that I think of it.
ptc_weather_2023 = ptc_weather[ptc_weather["year"] == 2023].copy()
ptc_weather_2023_dev = ptc_weather_2023.copy()
# Here's where we set the holidays (again). We could refactor this code to use the full ptc_weather dataset from the get-go haha
ptc_weather_2023_dev["is_holiday"] = ptc_weather_2023_dev["ds"].dt.date.map(
    lambda x : 1 if x in ee_holidays else 0
)
ptc_weather_2023_dev = ptc_weather_2023_dev.drop(
    columns=[
        "year", "month", "day", "hour",
        "in_sum", "out_sum", "y" #"day_of_week"
    ]
)

In [16]:
# This one has the ground truth, we use it to pass the true values and hence, evaluate
ptc_weather_2023_ground_truth = ptc_weather_2023.copy()
ptc_weather_2023_ground_truth["is_holiday"] = ptc_weather_2023_ground_truth["ds"].dt.date.map(
    lambda x : 1 if x in ee_holidays else 0
)
ptc_weather_2023_ground_truth = ptc_weather_2023_ground_truth.drop(
    columns=[
        "year", "month", "day", "hour",
        "in_sum", "out_sum" #, "day_of_week"
    ]
)
ptc_weather_2023_ground_truth.head(5)

,unique_id,ds,y,day_of_week,temp_mean,humidity_mean,precipitation_mean,rain_mean,snowfall_mean,snow_depth_mean,wind_speed_mean,cloud_cover_mean,cloud_cover_low_mean,cloud_cover_mid_mean,cloud_cover_high_mean,is_holiday
8733,all_sensors,2023-01-01 00:00:00,6420.0,6,5.5,97.0,0.2,0.2,0.0,0.02,22.9,100.0,100.0,100.0,97.0,1
8734,all_sensors,2023-01-01 01:00:00,5575.0,6,5.4,96.0,0.1,0.1,0.0,0.02,23.3,100.0,49.0,100.0,38.0,1
8735,all_sensors,2023-01-01 02:00:00,4818.0,6,5.1,96.0,0.1,0.1,0.0,0.02,22.2,100.0,32.0,100.0,0.0,1
8736,all_sensors,2023-01-01 03:00:00,4268.0,6,4.6,95.0,0.0,0.0,0.0,0.01,22.7,100.0,44.0,0.0,2.0,1
8737,all_sensors,2023-01-01 04:00:00,2984.0,6,3.8,94.0,0.0,0.0,0.0,0.01,20.9,93.0,5.0,0.0,92.0,1


In [17]:
ptc_weather_2023_ground_truth["ds"].loc[14000]

Timestamp('2023-08-08 11:00:00')

In [18]:
# Also, let me limit this to 5 months only
ptc_weather_2022 = ptc_weather_2022[ptc_weather_2022["ds"] > "2022-07-31"]
len(ptc_weather_2022)

3694

In [19]:
print(ptc_weather_2022.columns)
print("---")
print(ptc_weather_2023_dev.columns)
print("---")
print(ptc_weather_2023_ground_truth.columns)

Index(['unique_id', 'ds', 'y', 'day_of_week', 'temp_mean', 'humidity_mean',
       'precipitation_mean', 'rain_mean', 'snowfall_mean', 'snow_depth_mean',
       'wind_speed_mean', 'cloud_cover_mean', 'cloud_cover_low_mean',
       'cloud_cover_mid_mean', 'cloud_cover_high_mean', 'is_holiday'],
      dtype='object')
---
Index(['unique_id', 'ds', 'day_of_week', 'temp_mean', 'humidity_mean',
       'precipitation_mean', 'rain_mean', 'snowfall_mean', 'snow_depth_mean',
       'wind_speed_mean', 'cloud_cover_mean', 'cloud_cover_low_mean',
       'cloud_cover_mid_mean', 'cloud_cover_high_mean', 'is_holiday'],
      dtype='object')
---
Index(['unique_id', 'ds', 'y', 'day_of_week', 'temp_mean', 'humidity_mean',
       'precipitation_mean', 'rain_mean', 'snowfall_mean', 'snow_depth_mean',
       'wind_speed_mean', 'cloud_cover_mean', 'cloud_cover_low_mean',
       'cloud_cover_mid_mean', 'cloud_cover_high_mean', 'is_holiday'],
      dtype='object')


In [20]:
## Predicting with Forecast
# Since I'm training from scratch, I need to re-train every single time...

prediction_start_date = ptc_weather_2023_ground_truth["ds"].loc[14000]

window_size = len(ptc_weather_2022["ds"]) #So for this file I'm sharing with you, I've reduced the training window.
all_preds = []
hour_window = 8

for i in range(0, 6, 1):
    prediction_start_date = prediction_start_date + pd.Timedelta(hours=4)
    pass_gt_2023 = ptc_weather_2023_ground_truth[
        ptc_weather_2023_ground_truth["ds"] < prediction_start_date
    ] #Thus here, we get more data from 2023 (a new data point)

    pass_gt = pd.concat([ #We combine the training data + our new example
        ptc_weather_2022[(i*4):window_size],
        pass_gt_2023
    ])

    # Let's readjust the MAD outlier cleaning
    # Step 1: Calculate the Median for the time series
    median = pass_gt["y"].median()
    # Step 2: Calculate the median absolute deviation
    mad = (pass_gt["y"] - median).abs().median()
    # Step 3: The threshold is 3 times that
    threshold = 3 * mad
    # Then we simply find those values and set it to the median
    pass_gt_3mad = pass_gt.copy()
    pass_gt_3mad.loc[(pass_gt_3mad["y"] - median).abs() > threshold, "y"] = median

    #Move the future data up four hours.
    future_weather = ptc_weather_2023_dev.iloc[len(pass_gt_2023) : len(pass_gt_2023)+ hour_window]
    assert len(future_weather) == hour_window, f"Expected {hour_window} rows, got {len(future_weather)}"

    # Setup the model to be trained.
    sf = StatsForecast(
        models=[MSTL(
            season_length=[24, 24*7],  # adjust to your original config
            trend_forecaster=AutoARIMA()
        )],
        freq='1h',
        n_jobs=1,  # start with 1 to avoid multiprocessing issues
    )
    
    moving_forecasts = sf.forecast(
        h=hour_window,
        df=pass_gt_3mad,
        X_df = future_weather,
        level=[80, 95]
    )
    moving_forecasts['iteration'] = i
    moving_forecasts['cutoff'] = prediction_start_date
    all_preds.append(moving_forecasts)

In [26]:
all_preds_df = pd.concat(all_preds, ignore_index=True)
# We first get the ground truth to compare to...
actuals = ptc_weather_2023_ground_truth[['unique_id', 'ds', 'y']].copy()

# Then merge it to the predictions.
eval_df = all_preds_df.merge(actuals, on=['unique_id', 'ds'], how='left')

# Now to compute the metrics....
# Please do tell me if something's wrong in these calculations!
eval_df['error'] = eval_df["MSTL"] - eval_df['y']
eval_df['abs_error'] = eval_df['error'].abs()
print("MAE: ", eval_df['abs_error'].mean())
print("RMSE:", (eval_df['error']**2).mean()**0.5)
print("sMAPE:", (2 * eval_df['abs_error'] / (eval_df["MSTL"].abs() + eval_df['y'].abs())).mean())

# Let's print the performance per iteration.
print(eval_df.groupby('iteration')['abs_error'].mean())

MAE:  524.054822993155
RMSE: 673.0030426410974
sMAPE: 0.1622548074248793
iteration
0    341.623
1    411.342
2    274.425
3    803.936
4    845.116
5    467.887
Name: abs_error, dtype: float64


In [ ]:
MAE:  517.0127896656332
RMSE: 658.5174644728105
sMAPE: 0.1597609710511341
iteration
0    334.511
1    399.289
2    283.807
3    765.104
4    846.635
5    472.731
Name: abs_error, dtype: float64

So performance can vary, this was using five months, I've used one year of data too, but that takes longer.
Still, if you look at the idea of re-training every four hours then generating 8 hours of prediction, taking 2 minutes to retrain the model does not seem that long.